# Café Analytics - Descriptive Analysis

In [ ]:
import pandas as pd
import warnings
import numpy as np
from datetime import datetime, date, timedelta
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Ignore all warnings in output
warnings.filterwarnings('ignore')

In [ ]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

In [ ]:
# Load the CSV file into a pandas DataFrame and 
# parse the 'order_time' column as datetime
df = pd.read_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv',
    parse_dates=['order_time'],
)

## Revenue Trend Analysis

In [ ]:
df_revenue = df[
    ['order_id', 'customer_id', 'order_price', 'order_time']
].drop_duplicates().reset_index(drop=True)

In [ ]:
df_revenue.head()

In [ ]:
df_revenue['order_year'] = (
    df_revenue['order_time'].dt.year
)
df_revenue['order_month'] = (
    df_revenue['order_time'].dt.month
)
df_revenue['order_day'] = (
    df_revenue['order_time'].dt.day
)
df_revenue['order_dow'] = (
    df_revenue['order_time'].dt.dayofweek
)
df_revenue['order_hour'] = (
    df_revenue['order_time'].dt.hour
)

In [ ]:
df_revenue['FY'] = (
    df_revenue.apply(
        lambda x: 'FY' + str(x['order_year'])[-2:] if x['order_month'] < 7
        else 'FY' + str(x['order_year'] + 1)[-2:],
        axis=1,
    )
)

In [ ]:
df_revenue.head()

### Monthly Revenue Trend

In [ ]:
monthly = (
    df_revenue[
        ['order_year', 'order_month', 'order_price']
    ].groupby(['order_year', 'order_month']).sum()
    .reset_index(drop=False)
)

In [ ]:
monthly.head()

In [ ]:
# Plot monthly revenue trend
fig, axes = plt.subplots(nrows=2, figsize=(10, 8))

for year in sorted(monthly['order_year'].unique()):
    month_data = monthly[
        monthly['order_year'] == year
    ]
    Y = month_data['order_price']
    X = month_data['order_month']
    
    axes[1].plot(X, Y, label=year, marker='.')
    axes[0].bar(
        x=year,
        height=month_data['order_price'].sum(), 
        label=year,
    )

axes[0].legend(loc='upper left', bbox_to_anchor=(1.05, 1))
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Online Sales (AUD)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Online Sales (AUD)')
axes[1].set_xticks(range(1, 13, 1))
plt.show()

* The bar chart above illustrates a **consistent rise** in business online sales from 2020 to 2023.
* Online sales in 2020 and 2021 consistently outperformed their corresponding previous year across all months, showing a notable increase during the winter season (June to August) and peaking in September before decliningg.
* In 2022, online sales generally stabilized over the course of the year but stayed higher than in 2021, except during the period from August to October. 
* Similary, online sales in 2023 remained stable overall but generally exceeded those of 2022 throughout the year.
* However, neither 2022 nor 2023 reached the September peak achieved in 2020 and 2021.

Question to investigate:<br><br>
**1) What factors contributed to the surge in sales from June to September in 2020 and 2021?**<br>
**2) Why did sales in 2022 and 2023 fail to reach the same peak in September?**<br>

Possible reasons: 

The **COVID pandemic lockdowns** likely boosted online sales, making the 2020 and 2021 figures more reflective of total sales (including both online and in-store). In contrast, the stablized sales in 2022 and 2023 may represent online sales only, as in-store shopping resumed.

As seen in the 2020 and 2021 figures, the café's online sales typically rise during the colder months from June to September in Australia, followed by a decline as the weather warms.

**TBC...**

### Most Profitable Day of the Week

In [ ]:
total_revenue_per_day = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_dow', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_dow'
    ]).sum().reset_index(drop=False)
)

total_revenue_per_day.head()

In [ ]:
dow_revenue = (
    total_revenue_per_day[
        ['order_dow', 'order_price']
    ].groupby('order_dow').mean()
    .reset_index(drop=False)
)

dow_revenue.head()

In [ ]:
dow_revenue_by_year = (
    total_revenue_per_day[
        ['order_year', 'order_dow', 'order_price']
    ].groupby(['order_year', 'order_dow']).mean()
    .reset_index(drop=False)
)

dow_revenue_by_year.head()

In [ ]:
day_of_week = {
    0: 'Mon',
    1: 'Tue',
    2: 'Wed',
    3: 'Thu',
    4: 'Fri',
    5: 'Sat',
    6: 'Sun',
}

In [ ]:
dow_revenue_by_year_pivoted = dow_revenue_by_year.pivot(
    index='order_dow',
    columns='order_year',
    values='order_price',
).reset_index(drop=False)

dow_revenue_by_year_pivoted.head()

In [ ]:
dow_revenue

In [ ]:
# Plot average daily revenue over a week
fig, axes = plt.subplots(
    nrows=2, 
    figsize=(10, 8), 
    sharex=True, 
    sharey=True,
)

dow_revenue.plot.bar(
    x='order_dow', y='order_price', 
    color='#D3D0C9', ax=axes[0]
)

# Highlight the most profitable day of the week
top_dow = dow_revenue[
    (
        dow_revenue['order_price'] 
        == dow_revenue['order_price'].max() 
    )
]
axes[0].bar(
    x=top_dow['order_dow'],
    height=top_dow['order_price'],
    color='orange',
    width=0.5,
)

# Plot average daily revenue by year across a week
dow_revenue_by_year_pivoted.plot.bar(
    x='order_dow', ax=axes[1]
)

plt.xticks(
    ticks=list(day_of_week.keys()),
    labels=list(day_of_week.values()), 
    rotation=45,
)
plt.xlabel('')
axes[0].get_legend().remove()
axes[1].legend(loc='upper left', bbox_to_anchor=(1.05, 1))
axes[0].set_ylabel('Online Sales (AUD)')
axes[1].set_ylabel('Online Sales (AUD)')
plt.show()

As shown in the bar charts above, the average daily online sales generally remained steady from Mondays to Thursdays, but rose higher during Fridays and weekends, with a peak on **Saturdays**.

### Peak Revenue Hours

In [ ]:
df_revenue.head()

In [ ]:
# Group sales revenue by year, month, day and hour
total_revenue_per_hour = (
    df_revenue[
        ['order_year', 'order_month', 'order_day', 
         'order_hour', 'order_price']
    ].groupby([
        'order_year', 'order_month', 
        'order_day', 'order_hour'
    ]).sum().reset_index(drop=False)
)

In [ ]:
print(total_revenue_per_hour.shape)
total_revenue_per_hour.head()

In [ ]:
# Display operating hours
sorted(total_revenue_per_hour['order_hour'].unique())

The operating hours seem unusual for a café. Let's check whether there are any outliers (irregular business hours) among them.

#### Identify and Remove Outliers

In [ ]:
# Plot the frequency distribution of records across operating hours
fig, ax = plt.subplots()
sns.kdeplot(
    x=total_revenue_per_hour['order_hour'],
    bw_adjust=1.6,  
    color='gray',
    ax=ax,
)
ax.set_xlabel('Order Hour')
ax.set_ylabel('Frequency')
ax.set_xticks(
    range(
        0,
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

According to the frequency plot above, the operating hours in most days are between 5AM to 3PM. Let's confirm this using a boxplot and identify any potential outliers.

In [ ]:
# Draw a boxplot to identify outliers
fig, ax = plt.subplots()
sns.boxplot(
    y=total_revenue_per_hour['order_hour'],
    whis=1.5,    # Set the whiskers to extend up to 1.5 times the interquartile range (IQR)
    ax=ax,
)
plt.ylabel('Operating Hour')
plt.yticks(
    range(
        total_revenue_per_hour['order_hour'].min(),
        total_revenue_per_hour['order_hour'].max() + 1,
        1,
    )
)
plt.show()

As shown in the boxplot above, it appears that the business hours extending past 5 PM have been identified as outliers.

In [ ]:
# Calculate the 1st quartile
Q1 = total_revenue_per_hour['order_hour'].quantile(0.25)
# Calculate the 3rd quartile
Q3 = total_revenue_per_hour['order_hour'].quantile(0.75)
# Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Calculate the lower bound where the lower whisker ends
lower_bound = Q1 - 1.5 * IQR
# Calculate the upper bound where the upper whisker ends
upper_bound = Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

Data points falling below the lower bound or above the upper bound are considered **outliers**. Let's apply the lower and upper bounds to filter out irregular business hours (occurring on only a few days throughout the business history) and determine the actual operating hours.

In [ ]:
# Filter out irregular business hours
total_revenue_per_hour = total_revenue_per_hour[
    (total_revenue_per_hour['order_hour'] >= lower_bound)
    & (total_revenue_per_hour['order_hour'] <= upper_bound)
]

In [ ]:
# Print the actual operating hours
sorted(total_revenue_per_hour['order_hour'].unique())

After filtering out the irregular operating hours, we can see that the café operates from **5 AM to 5 PM** on a typical business day.

In [ ]:
# Calculate the average online sales revenue per hour
hourly_revenue = (
    total_revenue_per_hour[
        ['order_hour', 'order_price']
    ].groupby('order_hour').mean()
    .reset_index(drop=False)
)

hourly_revenue

In [ ]:
# Calculate the average revenue per hour across all days
avg_revenue_per_hour = total_revenue_per_hour['order_price'].mean()
print(avg_revenue_per_hour)

In [ ]:
# Plot average hourly revenue across hours
plt.bar(
    x=hourly_revenue['order_hour'],
    height=hourly_revenue['order_price'],
    color='#D3D0C9',
)
# Plot a constant line representing average revenue per 
# hour across all days
plt.plot(
    range(0, 24, 1),
    [avg_revenue_per_hour] * 24,
    linestyle='--',
    linewidth=0.5,
    c='black',
)

# Highlight the peak revenue hours
top_revenue = hourly_revenue[
    hourly_revenue['order_price'] > avg_revenue_per_hour
]

plt.bar(
    x=top_revenue['order_hour'],
    height=top_revenue['order_price'],
    color='orange',
)
plt.xlabel('Hour')
plt.ylabel('Online Sales (AUD)')
plt.xticks(range(0, 24, 1))
plt.show()

The peak revenue hours that drive online sales above average are typically between **7 AM and 10 AM**, with the highest sales occurring at **8AM**.